# 1. Cadrage & API

**Entreprise** : Boutique en ligne de goodies et cartes Pokémon.

**Besoin** : Piloter les stocks et actions marketing via l'analyse des Pokémon.

**Source** : PokéAPI ([https://pokeapi.co/api/v2/](https://pokeapi.co/api/v2/)).

**Données** : Identifiant, nom, poids, taille, XP de base, type.

---

**Github** : https://github.com/Geekoul/API-Pokemon

**Data studio** : https://datastudio.google.com/s/qPuVkZtImAI

# 2. Code Python

## 2.1 Collecte & Préparation (Python)

In [1]:
import pandas as pd
import requests

url = "https://pokeapi.co/api/v2/pokemon?limit=150"
response = requests.get(url)
data = response.json()["results"]

rows = []
for p in data:
  details = requests.get(p["url"]).json()
  rows.append({
      "id": details["id"],
      "name": details["name"],
      "height": details["height"],
      "weight": details["weight"],
      "base_experience": details["base_experience"],
      "type": details["types"][0]["type"]["name"],
  })

df = pd.DataFrame(rows)
df = df.drop_duplicates().dropna()  # Nettoyage

In [2]:
# vérifier l'éxecution de lAPI
if response.status_code == 200:
    data = response.json()
    print("Données récupérées !")
else:
    print("Erreur :", response.status_code)

Données récupérées !


## 3. Requêtes SQL (BigQuery)

In [15]:
# 1. Charger d'abord dans BigQuery
import pandas_gbq

pandas_gbq.to_gbq(
    df,
    "pokemon_db.raw_pokemon",
    project_id="pokemon-api-511811",
    if_exists="replace",
)

# 2. Exécuter la requête ensuite
query = """
    SELECT type, AVG(weight) as avg_weight, AVG(height) as avg_height 
    FROM `pokemon_db.raw_pokemon` 
    GROUP BY type
"""
df_result = pandas_gbq.read_gbq(query, project_id="pokemon-api-511811")

100%|██████████| 1/1 [00:00<00:00, 11037.64it/s]


Downloading: 100%|██████████|


In [16]:
#3. Top 5 XP

query_2 = """
    SELECT name, base_experience 
    FROM `pokemon-api-511811.pokemon_db.raw_pokemon` 
    ORDER BY base_experience DESC LIMIT 5
"""
df_top = pandas_gbq.read_gbq(query_2, project_id="pokemon-api-511811")

Downloading: 100%|██████████|


In [17]:
# 4. Comptage par type

query_3 = """
    SELECT type, COUNT(*) as total 
    FROM `pokemon-api-511811.pokemon_db.raw_pokemon` 
    GROUP BY type 
    ORDER BY total DESC
"""
df_count = pandas_gbq.read_gbq(query_3, project_id="pokemon-api-511811")

Downloading: 100%|██████████|
